In [0]:
%python

from datetime import datetime
import re
from collections import defaultdict

# Defines centralized S3 bucket name.

bucket = "retail-etl-dwh-lakehouse"

# incoming csv files
sftp_path    = f"s3://{bucket}/sftp/"

# latest processing files
raw_path     = f"s3://{bucket}/raw/"

# historical backup
archive_path = f"s3://{bucket}/archive/"

# Creates today’s date for dynamic folders
today = datetime.now().strftime("%d%m%Y")

print(f"Today: {today}")

# Reads all incoming files from S3
sftp_files = dbutils.fs.ls(sftp_path)
table_files = defaultdict(list)
print("\n=== Files in sftp/ ===")

for f in sftp_files:
    # tablename_src_timestamp.csv to identify latest file
    match = re.match(r"(.+_src)_(\d{14})\.csv", f.name)

    if match:

        table_name = match.group(1)

        timestamp = datetime.strptime(
            match.group(2),
            "%d%m%Y%H%M%S"
        )

        table_files[table_name].append(
            (timestamp, f.path, f.name)
        )

        print(f"✅ {f.name}")

    else:
        print(f"❌ Skipped: {f.name}")

# Deletes previous raw files before loading latest data for avoiding duplicate processing

print("\n=== Archiving OLD files ===")

archive_today = f"{archive_path}{today}/"

for table_name, files in table_files.items():

    # Sorts by timestamp descending and picks latest file
    sorted_files = sorted(
        files,
        key=lambda x: x[0],
        reverse=True
    )

    if len(sorted_files) > 1:

        # latest file stays in raw
        latest_file = sorted_files[0]

        # all remaining become archive
        old_files = sorted_files[1:]

        for old_file in old_files:

            source_path = old_file[1]
            file_name   = old_file[2]

            archive_dest = archive_today + file_name

            # prevent duplicate archive copies
            archive_exists = False

            try:

                existing_files = dbutils.fs.ls(archive_today)

                for ef in existing_files:

                    if ef.name == file_name:

                        archive_exists = True
                        break

            except:
                pass

            # copy only if not already archived
            if not archive_exists:

                dbutils.fs.cp(
                    source_path,
                    archive_dest
                )

                print(f"📦 Archived OLD file: {file_name}")

    else:

        print(
            f"ℹ️ No old files to archive for {table_name}"
        )

# CLEAR RAW FOLDER

print("\n=== Clearing raw/ ===")

try:

    raw_files = dbutils.fs.ls(raw_path)

    for f in raw_files:

        if f.name.endswith(".csv"):

            dbutils.fs.rm(f.path)

            print(f"🗑️ Removed: {f.name}")

except Exception as e:

    print("ℹ️ raw/ already empty")

# COPY LATEST FILES TO RAW

print("\n=== Copying latest files to raw/ ===")

for table_name, files in table_files.items():

    latest_file = sorted(
        files,
        key=lambda x: x[0],
        reverse=True
    )[0]

    source_path = latest_file[1]
    file_name   = latest_file[2]

    destination = raw_path + file_name

    dbutils.fs.cp(source_path, destination)

    print(f"✅ Latest copied: {file_name}")

# FINAL VALIDATION

print("\n=== raw/ contains ===")

for f in dbutils.fs.ls(raw_path):

    if f.name.endswith(".csv"):

        print(f"✅ {f.name}")

print("\n=== archive/ contains ===")

try:

    archive_folders = dbutils.fs.ls(archive_path)

    for folder in archive_folders:

        for f in dbutils.fs.ls(folder.path):

            if f.name.endswith(".csv"):

                print(f"📦 {folder.name}{f.name}")

except:

    print("ℹ️ archive/ empty")